# Integration & Quadrature

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caam37830-sci-comp/notes/blob/main/04_functions/integration.ipynb)

First, let's recall a few definitions.  An indefinite integral is an integral without bounds, and is defined up to a constant
\begin{equation}
\int x\, dx = \frac{x^2}{2} + C
\end{equation}
A definite integral has bounds, which are sometimes symbolic
\begin{equation}
\int_0^y 1\, dx = y
\end{equation}

In [ ]:
import numpy as np
from scipy.integrate import dblquad, fixed_quad, nquad, quad, simpson, trapezoid
import matplotlib.pyplot as plt

import sympy as sym

## Symbolic Integration

`sympy` includes a variety of functionality for integration.

In [ ]:
import sympy as sym
x, y = sym.symbols('x y')

You can compute indefinite integrals

In [ ]:
sym.integrate(1, x) 

In [ ]:
sym.integrate(x**2 + x - 1)

In [ ]:
sym.integrate(x*y, y)

If you want to compute definite integrals, you can pass in the bounds

In [ ]:
sym.integrate(1, (x, 0, 2))

In [ ]:
sym.integrate(sym.sin(x), (x, 0, sym.pi))

In [ ]:
sym.integrate(x*y, (y, 0, 1))

In [ ]:
sym.integrate(x, (x, 0, y))

SymPy also provides integral transforms such as the Fourier transform

In [ ]:
k = sym.Symbol('k')
sym.fourier_transform(sym.exp(-x**2), x, k)

Not every integral has a closed form.  Many elementary functions, such as $e^{-x^2}$ or $\sin(\sin x)$, have antiderivatives that cannot be written in terms of elementary functions.  SymPy writes some of these with special functions, here the error function $\operatorname{erf}$,

In [ ]:
sym.integrate(sym.exp(-x**2), (x, 0, 1))

and returns others unevaluated

In [ ]:
integral = sym.integrate(sym.sin(sym.sin(x)), (x, 0, 1))
integral

We can still get a number.  `evalf` computes one with a quadrature rule, which is the subject of the rest of this section.

In [ ]:
integral.evalf()

You can find more complete information in the [sympy documentation](https://docs.sympy.org/latest/modules/integrals/integrals.html)

## Numerical Integration (Quadrature)

Numerical integration is used to obtain definite integrals.

First, we should recall the definition of the Riemann integral:
\begin{equation}
\int_a^b f(x)\, dx = \lim_{n\to \infty} \sum_{i=1}^n f(x_i) (x_{i+1} - x_i)
\end{equation}
where the sequence $\{x_i\}$ is built for each $n$ so that $x_1 = a$ and $x_{i+1} - x_{i} = (b-a)/n$.  It is useful to think of how integrals compute area under a curve.  In this case, we are (approximately) computing this area as the sum of rectangles with height $f(x_i)$ and width $x_{i+1}-x_i$.  More generally, we can think of this as a weighted sum of function values
\begin{equation}
\sum_{i=1}^n f(x_i) w_i
\end{equation}
Which looks more like Lebesgue integration, where $w_i$ is the measure assigned to the point $x_i$.

In order to numerically integrate functions, we will take a finite sum of weighted function values, where the weights and function evaluation points are designed to give a good approximation to the integral.  This is known as [quadrature](https://en.wikipedia.org/wiki/Quadrature_(mathematics)) (instead of integration).

A quadrature rule with **nodes** $x_i$ and **weights** $w_i$ approximates
$$ \int_a^b f(x)\, dx \approx \sum_{i=1}^n w_i f(x_i). $$
Evaluating $f$ is usually the expensive part, so we compare rules by how small their error is for a given number $n$ of function evaluations.

### Trapezoidal Rule

One of the simplest quadrature rules is the trapezoidal rule, which approximates an area under a curve by trapezoids defined by the four points $(x_i,0), (x_{i+1},0), (x_i, f(x_i)), (x_{i+1}, f(x_{i+1}))$.  The area of this trapezoid is 
\begin{equation}
\frac{1}{2}\bigg(f(x_{i+1}) + f(x_i)\bigg)(x_{i+1} - x_i)
\end{equation}
If we have $a = x_0 \le x_1 \le ... \le x_{n} = b$ equally spaced with spacing $h = (b-a)/n$, then the quadrature weights for $x_i$ are $w_i = h$ if $i\notin \{0,n\}$, and $w_i= h/2$ if $i \in \{0,n\}$:
$$ \int_a^b f(x)\, dx \approx T_h = h\left(\tfrac12 f(x_0) + f(x_1) + \cdots + f(x_{n-1}) + \tfrac12 f(x_n)\right). $$

You can use the trapezoidal rule in `scipy` with [`scipy.integrate.trapezoid`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.trapezoid.html#scipy.integrate.trapezoid)

In [ ]:
import numpy as np
from scipy.integrate import trapezoid

x = np.linspace(0,1,100)
y = x**2
trapezoid(y, x) # actual answer is 1/3

The exact answer is $1/3$, so the error is about $1.7 \times 10^{-5}$.  Where does it come from?  On each interval $[x_i, x_{i+1}]$, the area of the trapezoid is the integral of the straight line through $(x_i, f(x_i))$ and $(x_{i+1}, f(x_{i+1}))$.  The error of linear interpolation is $\frac12 f''(\eta)(x - x_i)(x - x_{i+1})$ for some $\eta$ between $x_i$ and $x_{i+1}$ that depends on $x$, and integrating it over the interval gives $-\frac{h^3}{12} f''(\xi_i)$ for some $\xi_i$ in the interval.  Adding up the $n = (b-a)/h$ intervals gives
$$ \int_a^b f(x)\, dx - T_h = -\frac{(b-a)\, h^2}{12} f''(\xi) $$
for some $\xi \in [a, b]$.  A local error of $O(h^3)$ on each of $O(1/h)$ intervals adds up to a global error of $O(h^2)$, so the trapezoidal rule is **second order**: halving $h$ divides the error by about 4.  For $f(x) = x^2$ the second derivative is the constant 2, so the trapezoidal rule overestimates the integral by exactly $h^2/6 \approx 1.7\times 10^{-5}$ when $h = 1/99$, as we saw.

### Simpson's Rule

The trapezoidal rule integrates the piecewise linear interpolant of $f$.  Integrating an interpolant of higher degree gives a more accurate rule.  The quadratic that interpolates $f$ at the ends and the midpoint of an interval of width $2h$ integrates to **Simpson's rule**
$$ \int_{x_{i-1}}^{x_{i+1}} f(x)\, dx \approx \frac{h}{3}\bigl(f(x_{i-1}) + 4 f(x_i) + f(x_{i+1})\bigr). $$
Applying it to consecutive pairs of intervals (so $n$ must be even) gives the weights $\frac h3 (1, 4, 2, 4, 2, \dots, 2, 4, 1)$.  Rules like these, which integrate a polynomial interpolant on equally spaced nodes, are called **Newton-Cotes** rules.

Simpson's rule turns out to be exact for cubics as well as for quadratics: on each pair of intervals, any cubic is a quadratic plus a multiple of $(x - x_i)^3$, and the rule integrates $(x - x_i)^3$ exactly (to zero), because it is odd about the midpoint $x_i$.  The error is
$$ \int_a^b f(x)\, dx - S_h = -\frac{(b-a)\, h^4}{180} f^{(4)}(\xi), $$
so Simpson's rule is fourth order.  It is available as [`scipy.integrate.simpson`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.simpson.html).

In [ ]:
x = np.linspace(0, 1, 101)      # 100 intervals, h = 0.01
for k in [2, 3, 4]:
    print(f"x^{k}: Simpson's rule error {simpson(x**k, x=x) - 1/(k + 1):.1e}")

For $x^4$ the error formula gives $\frac{h^4}{180} \cdot 4! \approx 1.3 \times 10^{-9}$, which is what we see.

Doubling the number of intervals should divide the error by 4 for the trapezoidal rule and by 16 for Simpson's rule.  Let's check on $\int_0^1 e^x\, dx = e - 1$.

In [ ]:
F = np.e - 1                     # the integral of e^x over [0, 1]
for n in [10, 20, 40, 80]:       # number of intervals
    x = np.linspace(0, 1, n + 1)
    print(f"n = {n:2d}:  trapezoid error {trapezoid(np.exp(x), x=x) - F:.2e},  "
          f"Simpson error {simpson(np.exp(x), x=x) - F:.2e}")

### Gaussian Quadrature

Newton-Cotes rules fix the nodes and choose the weights.  Gaussian quadrature chooses both, to make the rule exact for polynomials of the highest possible degree.  For smooth functions it is far more accurate than the rules above for the same number of function evaluations.  This is what `fixed_quad` computes.  If you want to integrate to a fixed tolerance instead of with a fixed number of points, use `quad`, which we'll see more of below; it is built on Gaussian quadrature too.

#### Weights from nodes

Suppose that $n$ distinct nodes $x_1, \dots, x_n$ in $[a, b]$ are given.  There is exactly one polynomial of degree at most $n - 1$ that interpolates $f$ at the nodes, and we can write it in the Lagrange form from [Interpolation](interpolation.ipynb),
$$ p(x) = \sum_{i=1}^n f(x_i)\, \ell_i(x), \qquad \ell_i(x) = \prod_{j \ne i} \frac{x - x_j}{x_i - x_j}. $$
Integrating $p$ instead of $f$ gives a quadrature rule
$$ \int_a^b f(x)\, dx \approx \int_a^b p(x)\, dx = \sum_{i=1}^n w_i f(x_i), \qquad w_i = \int_a^b \ell_i(x)\, dx. $$
If $f$ is a polynomial of degree at most $n - 1$, then $p = f$ and the rule is exact.  Conversely, a rule that is exact for $1, x, \dots, x^{n-1}$ must have these weights, because they are the only solution of the $n$ linear equations
$$ \sum_{i=1}^n w_i\, x_i^k = \int_a^b x^k\, dx = \frac{b^{k+1} - a^{k+1}}{k+1}, \qquad k = 0, 1, \dots, n-1, $$
whose matrix is a (transposed) Vandermonde matrix.  Rules with these weights are called **interpolatory**.  For three equally spaced nodes on $[0, 1]$ we get Simpson's rule back:

In [ ]:
def interpolatory_weights(nodes, a, b):
    """
    weights of the quadrature rule on [a, b] with the given nodes
    that integrates polynomials of degree < len(nodes) exactly
    """
    k = np.arange(len(nodes))
    V = nodes[np.newaxis, :] ** k[:, np.newaxis]       # V[k, i] = x_i^k
    moments = (b**(k + 1) - a**(k + 1)) / (k + 1)      # integrals of x^k over [a, b]
    return np.linalg.solve(V, moments)

interpolatory_weights(np.array([0, 0.5, 1]), 0, 1)     # Simpson's rule: 1/6, 4/6, 1/6

(Vandermonde matrices become badly conditioned as $n$ grows, so this is a way to understand the weights rather than a good way to compute them.)

#### Choosing the nodes

With $n$ nodes and $n$ weights there are $2n$ numbers to choose, so we can hope to integrate $1, x, \dots, x^{2n-1}$ exactly: $2n$ equations for $2n$ unknowns.  The equations are now nonlinear, because the nodes are unknown too.  For $n = 2$ on $[-1, 1]$, SymPy can solve them.

In [ ]:
s = sym.Symbol('s')
x1, x2, w1, w2 = sym.symbols('x1 x2 w1 w2', real=True)
moment_equations = [w1 * x1**k + w2 * x2**k - sym.integrate(s**k, (s, -1, 1)) for k in range(4)]
sym.solve(moment_equations, [x1, x2, w1, w2], dict=True)

Up to the order of the nodes there is one solution: nodes $\pm 1/\sqrt 3$, both with weight 1.  This two-point rule integrates every cubic on $[-1, 1]$ exactly, the same degree as Simpson's rule with one evaluation fewer.

Degree $2n - 1$ is the best possible.  The polynomial $q(x) = (x - x_1)^2 (x - x_2)^2 \cdots (x - x_n)^2$ has degree $2n$ and a positive integral, but any rule with nodes $x_1, \dots, x_n$ gives $\sum_i w_i q(x_i) = 0$.  Solving nonlinear equations quickly becomes hopeless as $n$ grows, though.  The structure that gives the nodes for every $n$ is orthogonality.

#### Orthogonal polynomials

Functions on $[-1, 1]$ have the inner product
$$ \langle f, g \rangle = \int_{-1}^1 f(x)\, g(x)\, dx. $$
Applying Gram-Schmidt to $1, x, x^2, \dots$ gives the **Legendre polynomials** $P_0, P_1, P_2, \dots$, where $P_n$ has degree $n$ and is orthogonal to every polynomial of degree less than $n$.  With the usual normalization $P_n(1) = 1$,
$$ P_0(x) = 1, \quad P_1(x) = x, \quad P_2(x) = \tfrac12 (3x^2 - 1), \quad P_3(x) = \tfrac12 (5x^3 - 3x), \quad \dots $$
The roots of $P_2$ are $\pm 1/\sqrt 3$, the nodes we just found.  That is not a coincidence.

**Theorem.**  Let $x_1, \dots, x_n$ be the roots of $P_n$ (we'll see below that they are real and distinct), and let $w_i = \int_{-1}^1 \ell_i(x)\, dx$ be the interpolatory weights for these nodes.  Then
$$ \sum_{i=1}^n w_i f(x_i) = \int_{-1}^1 f(x)\, dx $$
for every polynomial $f$ of degree at most $2n - 1$.

*Proof.*  Divide $f$ by $P_n$ with remainder: $f = q P_n + r$, where $q$ and $r$ have degree at most $n - 1$.  Since $P_n$ is orthogonal to $q$,
$$ \int_{-1}^1 f\, dx = \int_{-1}^1 q P_n\, dx + \int_{-1}^1 r\, dx = \int_{-1}^1 r\, dx, $$
and since $P_n(x_i) = 0$,
$$ \sum_{i=1}^n w_i f(x_i) = \sum_{i=1}^n w_i\, q(x_i) P_n(x_i) + \sum_{i=1}^n w_i\, r(x_i) = \sum_{i=1}^n w_i\, r(x_i). $$
The two right-hand sides are equal, because the rule is interpolatory and $r$ has degree at most $n - 1$. $\square$

The argument also runs backwards, so no other nodes work.  If an $n$-point rule is exact up to degree $2n - 1$, then the node polynomial $\omega(x) = (x - x_1) \cdots (x - x_n)$ satisfies $\int_{-1}^1 \omega\, q\, dx = \sum_i w_i\, \omega(x_i)\, q(x_i) = 0$ for every $q$ of degree at most $n - 1$.  So $\omega$ is orthogonal to all polynomials of lower degree, which makes it a multiple of $P_n$.

Two more facts make the rule work well in practice.

* **The nodes are real, distinct, and lie in $(-1, 1)$.**  Suppose $P_n$ changed sign at only $m < n$ points $z_1, \dots, z_m$ in $(-1, 1)$.  Then $P_n(x)(x - z_1) \cdots (x - z_m)$ would not change sign on $[-1, 1]$, so its integral could not be zero.  But the integral is zero by orthogonality.  So $P_n$ changes sign at $n$ points in $(-1, 1)$, and these are all of its roots.
* **The weights are positive.**  The polynomial $\ell_i^2$ has degree $2n - 2$, so the rule integrates it exactly, and $\ell_i(x_j)$ is 1 for $j = i$ and 0 otherwise.  So $w_i = \sum_j w_j \ell_i(x_j)^2 = \int_{-1}^1 \ell_i(x)^2\, dx > 0$.  The weights also add up to $\int_{-1}^1 1\, dx = 2$, so errors $\delta_i$ in the values $f(x_i)$ are never amplified: $\bigl|\sum_i w_i\, \delta_i\bigr| \le 2\max_i |\delta_i|$.  High-order Newton-Cotes rules have large weights of both signs, which is one reason they are not used (see the exercises).

Let's check all this with NumPy's [`numpy.polynomial.legendre`](https://numpy.org/doc/stable/reference/routines.polynomials.legendre.html) module.  `legendre.leggauss(n)` returns the nodes and weights of the $n$-point Gauss-Legendre rule.

In [ ]:
from numpy.polynomial import legendre

n = 5
nodes, weights = legendre.leggauss(n)
print("roots of P_5 :", np.sort(legendre.Legendre.basis(n).roots()))
print("Gauss nodes  :", nodes)
print("weights      :", weights)
print("interpolatory weights for these nodes:", interpolatory_weights(nodes, -1, 1))
print("sum of the weights:", weights.sum())

In [ ]:
s = np.linspace(-1, 1, 400)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for n in range(1, 6):
    axs[0].plot(s, legendre.Legendre.basis(n)(s), label=f'$P_{n}$')
axs[0].axhline(0, color='gray', lw=0.5)
axs[0].set_ylim(-1.1, 1.5)
axs[0].legend(ncol=5, loc='upper center')
axs[0].set_title('Legendre polynomials')
for n in [5, 10, 20]:
    nodes, weights = legendre.leggauss(n)
    axs[1].plot(nodes, weights, 'o-', ms=4, label=f'$n = {n}$')
axs[1].set_xlabel('node $x_i$')
axs[1].set_ylabel('weight $w_i$')
axs[1].legend()
axs[1].set_title('Gauss-Legendre nodes and weights')
plt.show()

The nodes cluster towards the ends of the interval, like the Chebyshev points in [Interpolation](interpolation.ipynb), and the weights are smaller there.

#### Computing the nodes and weights

`leggauss` does not solve nonlinear equations either.  The Legendre polynomials satisfy the three-term recurrence
$$ (k+1)\, P_{k+1}(x) = (2k+1)\, x P_k(x) - k\, P_{k-1}(x), $$
which for the normalized polynomials $\hat P_k = \sqrt{k + 1/2}\, P_k$ (so that $\langle \hat P_j, \hat P_k \rangle$ is 1 for $j = k$ and 0 otherwise) reads
$$ x\, \hat P_k(x) = \beta_k \hat P_{k-1}(x) + \beta_{k+1} \hat P_{k+1}(x), \qquad \beta_k = \frac{k}{\sqrt{4k^2 - 1}}, $$
with $\hat P_{-1} = 0$.  Collecting the equations for $k = 0, 1, \dots, n-1$ into one equation for the vector $\mathbf p(x) = \bigl(\hat P_0(x), \dots, \hat P_{n-1}(x)\bigr)^\top$ gives
$$ x\, \mathbf p(x) = J\, \mathbf p(x) + \beta_n \hat P_n(x)\, \mathbf e_n, \qquad
J = \begin{pmatrix} 0 & \beta_1 & & \\ \beta_1 & 0 & \ddots & \\ & \ddots & \ddots & \beta_{n-1} \\ & & \beta_{n-1} & 0 \end{pmatrix}, $$
where $\mathbf e_n$ is the last standard basis vector.  At a root $x_i$ of $P_n$ the last term vanishes, and $J\, \mathbf p(x_i) = x_i\, \mathbf p(x_i)$.  So **the Gauss nodes are the eigenvalues of the symmetric tridiagonal matrix $J$**, with eigenvectors $\mathbf p(x_i)$.

The weights come from the eigenvectors.  The rule integrates the products $\hat P_j \hat P_k$ exactly, since their degree is at most $2n - 2$, so
$$ \sum_{i=1}^n w_i\, \hat P_j(x_i)\, \hat P_k(x_i) = \langle \hat P_j, \hat P_k \rangle = \begin{cases} 1 & j = k \\ 0 & j \ne k. \end{cases} $$
This says that the $n \times n$ matrix with columns $\sqrt{w_i}\, \mathbf p(x_i)$ is orthogonal.  Its columns are therefore the unit eigenvectors $\mathbf v_i$ of $J$ (up to sign), and comparing first entries, $(\mathbf v_i)_1 = \pm\sqrt{w_i}\, \hat P_0 = \pm\sqrt{w_i/2}$.  So
$$ w_i = 2\, (\mathbf v_i)_1^2. $$
This is the **Golub-Welsch algorithm**: a single symmetric tridiagonal eigenvalue problem gives all the nodes and weights, stably.  (NumPy's `leggauss` computes the nodes as the eigenvalues of this matrix, which it calls `legendre.legcompanion`, and then refines them with a step of Newton's method.)

In [ ]:
def gauss_legendre(n):
    """
    nodes and weights of the n-point Gauss-Legendre rule on [-1, 1], by Golub-Welsch
    """
    k = np.arange(1, n)
    beta = k / np.sqrt(4 * k**2 - 1)
    J = np.diag(beta, 1) + np.diag(beta, -1)
    nodes, V = np.linalg.eigh(J)      # eigenvalues in increasing order, unit eigenvectors
    weights = 2 * V[0, :]**2
    return nodes, weights

nodes, weights = gauss_legendre(20)
nodes_np, weights_np = legendre.leggauss(20)
print("largest difference from leggauss, nodes  :", np.abs(nodes - nodes_np).max())
print("largest difference from leggauss, weights:", np.abs(weights - weights_np).max())

#### Gauss-Legendre quadrature on $[a, b]$

To integrate over $[a, b]$, substitute $x = \frac{a + b}{2} + \frac{b - a}{2} t$, which maps $t \in [-1, 1]$ onto $x \in [a, b]$:
$$ \int_a^b f(x)\, dx = \frac{b - a}{2} \int_{-1}^1 f\Bigl(\frac{a + b}{2} + \frac{b - a}{2}\, t\Bigr)\, dt \approx \frac{b - a}{2} \sum_{i=1}^n w_i\, f\Bigl(\frac{a + b}{2} + \frac{b - a}{2}\, t_i\Bigr), $$
where $t_i$ and $w_i$ are the nodes and weights on $[-1, 1]$.  The substitution is linear, so the rule on $[a, b]$ is still exact up to degree $2n - 1$.

In [ ]:
def gauss_quad(f, a, b, n):
    """
    n-point Gauss-Legendre approximation to the integral of f over [a, b]
    """
    t, w = gauss_legendre(n)
    x = (a + b) / 2 + (b - a) / 2 * t
    return (b - a) / 2 * np.sum(w * f(x))

In [ ]:
from scipy.integrate import fixed_quad, quad

f = lambda x: x**10

`fixed_quad` uses $n$-point Gauss-Legendre quadrature (the default is $n = 5$).  An $n$-point rule integrates polynomials of degree up to $2n - 1$ exactly, so $n = 6$ is enough for $x^{10}$.

In [ ]:
fixed_quad(f, 0, 1, n=6) # n is the number of nodes

The return tuple is the computed result and `None`.  Our `gauss_quad` gives the same answer, and with one node fewer the rule is no longer exact:

In [ ]:
for n in [5, 6]:
    print(f"n = {n}: gauss_quad error {gauss_quad(f, 0, 1, n) - 1/11:+.1e}, "
          f"fixed_quad error {fixed_quad(f, 0, 1, n=n)[0] - 1/11:+.1e}")

#### How fast does it converge?

For smooth functions, Gaussian quadrature converges much faster than the trapezoidal rule or Simpson's rule.  Here are the errors of all three against the number of function evaluations for two functions on $[-1, 1]$: $e^x$, and the Runge function $1/(1 + 25x^2)$ from [Interpolation](interpolation.ipynb).

In [ ]:
tests = {
    "$e^x$": (np.exp, np.e - 1 / np.e),
    "$1/(1 + 25x^2)$": (lambda x: 1 / (1 + 25 * x**2), 2 / 5 * np.arctan(5)),
}
ns = np.arange(3, 120, 2)    # odd numbers of points, so that Simpson's rule has an even number of intervals

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, (g, exact)) in zip(axs, tests.items()):
    err_trap, err_simp, err_gauss = [], [], []
    for n in ns:
        x = np.linspace(-1, 1, n)
        err_trap.append(abs(trapezoid(g(x), x=x) - exact))
        err_simp.append(abs(simpson(g(x), x=x) - exact))
        err_gauss.append(abs(gauss_quad(g, -1, 1, n) - exact))
    ax.loglog(ns, err_trap, '.-', label='trapezoid')
    ax.loglog(ns, err_simp, '.-', label='Simpson')
    ax.loglog(ns, np.maximum(err_gauss, 1e-17), '.-', label='Gauss-Legendre')   # a few errors are exactly 0
    ax.set_title(name)
    ax.set_xlabel('number of function evaluations $n$')
axs[0].set_ylabel('error')
axs[0].legend()
plt.show()

For $e^x$, the trapezoidal and Simpson errors lie on straight lines with slopes $-2$ and $-4$ on these log-log axes, the rates we derived.  The Gauss-Legendre error falls much faster: for smooth functions it decreases *exponentially* in $n$.  For $e^x$, seven points give an error of about $2 \times 10^{-15}$.  To get that close, Simpson's rule would need about 3000 points, and the trapezoidal rule about $2 \times 10^7$.

The Runge function is harder.  The Gauss-Legendre error still decreases exponentially, but more slowly, and below about 25 points the trapezoidal rule is actually more accurate.  The exponential rate depends on how far from the interval $f$ can be continued as an analytic function into the complex plane: $1/(1 + 25x^2)$ has poles at $x = \pm i/5$, close to the interval, while $e^x$ has no singularities at all.  Note also that polynomial interpolation of the Runge function at equally spaced points diverges, while Gaussian quadrature converges, as it does for every continuous $f$, thanks to its positive weights.

#### Other weight functions

The same construction works for integrals $\int_a^b f(x)\, \rho(x)\, dx$ with a fixed positive weight function $\rho$: take the polynomials that are orthogonal for the inner product $\langle f, g \rangle = \int_a^b f(x)\, g(x)\, \rho(x)\, dx$.  The proofs above go through word for word, so the rule is exact whenever $f$ is a polynomial of degree at most $2n - 1$, and the Golub-Welsch algorithm works with the recurrence coefficients of the new polynomials.  The weight can be singular and the interval can be infinite, as long as $f$ itself is smooth.  The classical cases are in `scipy.special`:

| weight $\rho(x)$ | interval | orthogonal polynomials | nodes and weights |
|:--|:--|:--|:--|
| $1$ | $[-1, 1]$ | Legendre | `roots_legendre` |
| $1/\sqrt{1 - x^2}$ | $[-1, 1]$ | Chebyshev | `roots_chebyt` |
| $e^{-x}$ | $[0, \infty)$ | Laguerre | `roots_laguerre` |
| $e^{-x^2}$ | $(-\infty, \infty)$ | Hermite | `roots_hermite` |

For example, ten Gauss-Hermite points give $\int_{-\infty}^\infty e^{-x^2} \cos x\, dx = \sqrt{\pi}\, e^{-1/4}$ to about machine precision:

In [ ]:
from scipy.special import roots_hermite

nodes, weights = roots_hermite(10)
np.sum(weights * np.cos(nodes)) - np.sqrt(np.pi) * np.exp(-1/4)

#### Error estimates: Gauss-Kronrod rules

A quadrature routine should also tell us how accurate its answer is.  The obvious approach, comparing the $n$-point and $2n$-point Gauss rules, wastes the first $n$ evaluations, because the Gauss nodes for different $n$ hardly overlap.  **Gauss-Kronrod** rules avoid the waste.  They add $n + 1$ nodes to the $n$ Gauss nodes, and choose the new nodes and all $2n + 1$ weights to make the degree as high as possible (it comes out as $3n + 1$ or more).  The difference between the Gauss result and the Kronrod result estimates the error, and costs only $n + 1$ extra evaluations.

### Adaptive Quadrature

[`quad`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html#scipy.integrate.quad) wraps the Fortran library [QUADPACK](https://en.wikipedia.org/wiki/QUADPACK).  It integrates up to a fixed tolerance, adaptively choosing where to evaluate the function

In [ ]:
quad(f, 0, 1)

The return tuple is the computed result and an estimate of the absolute error.

On a finite interval, QUADPACK estimates the error on each subinterval by comparing a 10-point Gauss rule with a 21-point Gauss-Kronrod rule that reuses the same 10 points, and keeps subdividing the subintervals with the largest error until the estimate meets the tolerance.

To see the adaptivity at work, let's record where `quad` evaluates a function with a sharp peak at $x = 0.3$.  With `full_output=True`, `quad` also returns a dictionary of diagnostics, including the number of function evaluations `neval`, the number of subintervals `last`, and their left end points `alist`.

In [ ]:
evaluated = []       # every x at which quad calls the integrand

def peak(x):
    evaluated.append(x)
    return 1 / ((x - 0.3)**2 + 1e-4)

val, err, info = quad(peak, 0, 1, full_output=True)
exact = 100 * (np.arctan(70) + np.arctan(30))
print(f"error {val - exact:.1e}, estimated error {err:.1e}, "
      f"{info['neval']} evaluations on {info['last']} subintervals")

x = np.linspace(0, 1, 2000)
fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
ax[0].plot(x, 1 / ((x - 0.3)**2 + 1e-4))
for left in info['alist'][:info['last']]:      # boundaries of the final subintervals
    ax[0].axvline(left, color='gray', lw=0.5, ls='--')
ax[0].set_ylabel('$f(x)$')
ax[1].plot(evaluated, np.zeros(len(evaluated)), '|', ms=15)
ax[1].set_yticks([])
ax[1].set_xlabel('$x$')
ax[1].set_title('where quad evaluated $f$', fontsize=10)
plt.tight_layout()
plt.show()

`quad` bisected the subinterval containing the peak again and again, and spent most of its evaluations near it, while a single 21-point rule on $[0.5, 1]$ was enough for the smooth part.  The actual error is far smaller than the estimate.  That is typical: the estimate is built from the difference between the 10-point Gauss and the 21-point Kronrod results, but `quad` returns the Kronrod result, which is much more accurate.

`quad` stops when the estimated error is below `max(epsabs, epsrel * abs(result))`, where both tolerances default to $1.49 \times 10^{-8}$, or when it has made `limit=50` subintervals, in which case it issues an `IntegrationWarning`.

### Infinite Intervals

`quad` can also integrate over unbounded intervals, such as $\int_0^\infty e^{-x^2}\, dx = \sqrt{\pi}/2 \approx 0.8862269$.  QUADPACK maps $[0, \infty)$ onto $(0, 1]$ with the substitution $x = (1 - t)/t$, and then proceeds as before, with a 15-point Gauss-Kronrod rule.

In [ ]:
f = lambda x: np.exp(-x**2)

quad(f, 0, np.inf)

### Functions with Discontinuities

Discontinuities can cause problems with any sort of quadrature scheme that assumes functions are continuous.  You can specify problematic points with `points`

In [ ]:
import matplotlib.pyplot as plt

f = lambda x : np.floor(x**2)

x = np.linspace(0,2,100)
plt.plot(x, f(x))
plt.show()

In [ ]:
quad(f, 0, 2)

In [ ]:
quad(f, 0, 2, points=np.sqrt(np.arange(5)))

The integrand is constant between the jumps at $1, \sqrt 2, \sqrt 3$, so the exact value is $(\sqrt2 - 1) + 2(\sqrt3 - \sqrt2) + 3(2 - \sqrt3) = 5 - \sqrt2 - \sqrt3$.  Without the breakpoints, `quad` has to locate the jumps by bisection, which costs many more evaluations and still leaves an error of about $2 \times 10^{-9}$.  With them, `quad` integrates each piece separately, and on each piece the 21-point rule is exact.

In [ ]:
exact = 5 - np.sqrt(2) - np.sqrt(3)
for label, points in [("without points", None), ("with points   ", np.sqrt(np.arange(5)))]:
    val, err, info = quad(f, 0, 2, points=points, full_output=True)
    print(f"{label}: error {val - exact:+.1e} with {info['neval']} evaluations")

### Functions with Singularities

An integrable singularity at an end of the interval is usually not a problem.  For example, $\int_0^1 x^{-1/2}\, dx = 2$:

In [ ]:
quad(lambda x: 1 / np.sqrt(x), 0, 1)

QUADPACK never evaluates $f$ at the end points, and it speeds up the repeated bisection towards the singularity with an extrapolation method (Wynn's epsilon algorithm).

A singularity inside the interval needs more care.  If it is integrable, you can split the interval at the singularity, or pass its location in `points`.  If you also know the form of the singularity, you can pass it to `quad` as a weight function.  Depending on the type of singularity, you can pass in different arguments into the `weight` and `wvar` parameters of `quad`.  See [the documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html#scipy.integrate.quad) for details.

For example, let's say we want to compute

\begin{equation}
\int_{-1}^2 \frac{1}{x}\,dx
\end{equation}

Strictly speaking, this integral diverges: $\int_{\epsilon}^2 \frac{1}{x}\,dx$ and $\int_{-1}^{-\epsilon} \frac{1}{x}\,dx$ both blow up as $\epsilon \to 0$.  What we can compute is its **Cauchy principal value**, which cuts out an interval around the singularity symmetrically

\begin{equation}
\mathrm{p.v.}\int_{-1}^2 \frac{1}{x}\,dx = \lim_{\epsilon \to 0^+} \left( \int_{-1}^{-\epsilon} \frac{1}{x}\,dx + \int_{\epsilon}^2 \frac{1}{x}\,dx \right)
\end{equation}

In [ ]:
f = lambda x : 1 / x

x = np.linspace(-1,2,300) # 300 points, so that x = 0 is not one of them
plt.plot(x, f(x))
plt.ylim(-20, 20)
plt.show()

The integrand is odd, so the integrals over $[-1,-\epsilon]$ and $[\epsilon,1]$ cancel and

\begin{equation}
\mathrm{p.v.}\int_{-1}^2 \frac{1}{x}\,dx = \int_1^2 \frac{1}{x}\,dx = \log(2) - \log(1) = \log(2)
\end{equation}

The `'cauchy'` weight will calculate the principal value of the integral of $$\frac{f(x)}{x - {\tt wvar}}$$ (i.e. we will weight the function $f$ by $\frac{1}{x - {\tt wvar}}$).  The name refers to the Cauchy principal value, not the Cauchy distribution.

So we can use `weight='cauchy'` with `wvar=0` on the function `f(x) = 1` to compute our integral.


In [ ]:
quad(lambda x : 1, -1, 2, weight='cauchy', wvar=0) # multiply function by 1/x and do integration

In [ ]:
np.log(2)

## Multivariate Integration

You can integrate over functions of multiple variables using
* [`dblquad`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.dblquad.html#scipy.integrate.dblquad) for 2 variables
* [`tplquad`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.tplquad.html#scipy.integrate.tplquad) for 3 variables
* [`nquad`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.nquad.html#scipy.integrate.nquad) for `n` variables

`nquad` integrates over a hyper cube defined by ranges in the second argument. `dblquad` and `tplquad` can allow you to integrate over more complicated domains.

In [ ]:
from scipy.integrate import nquad

f = lambda x, y, z : np.exp( -x**2 - y**2 - z**2)

nquad(f, ((-3,3), (-3,3), (-3,3)))

The exact value is $\bigl(\sqrt{\pi}\, \operatorname{erf}(3)\bigr)^3 \approx 5.567959$.

`dblquad(f, a, b, gfun, hfun)` computes
$$ \int_a^b \int_{g(x)}^{h(x)} f(y, x)\, dy\, dx. $$
Watch the order of the arguments: the integrand takes the *inner* variable first.  Here is $\int_0^1 \int_0^x x y^2\, dy\, dx = 1/15$, the integral of $x y^2$ over the triangle $0 \le y \le x \le 1$, done correctly and with the arguments swapped.

In [ ]:
from scipy.integrate import dblquad

print(dblquad(lambda y, x: x * y**2, 0, 1, 0, lambda x: x))    # correct: 1/15 = 0.0667
print(dblquad(lambda x, y: x * y**2, 0, 1, 0, lambda x: x))    # swapped: this integrates y x^2, giving 1/10

### Cost in Higher Dimensions

`nquad` applies `quad` recursively: every evaluation of the integrand of the outer integral is itself an adaptive integral over the inner variables.  So the costs multiply.

In [ ]:
from scipy.special import erf

val, err, info = nquad(f, ((-3, 3), (-3, 3), (-3, 3)), full_output=True)
print(f"error {val - (np.sqrt(np.pi) * erf(3))**3:.1e} with {info['neval']} evaluations")

For a smooth function on a box, a **tensor-product** Gauss rule is much cheaper.  With nodes $x_i$ and weights $w_i$ in each direction, its nodes are all the points $(x_i, x_j, x_k)$, with weights $w_i w_j w_k$.

In [ ]:
t, w = legendre.leggauss(20)
x, w = 3 * t, 3 * w                     # the 20-point Gauss-Legendre rule on [-3, 3]
X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
W = w[:, None, None] * w[None, :, None] * w[None, None, :]
print(f"error {np.sum(W * f(X, Y, Z)) - (np.sqrt(np.pi) * erf(3))**3:.1e} with {W.size} evaluations")

Still, a tensor-product rule with $n$ points per direction needs $n^d$ evaluations in $d$ dimensions: 20 points per direction in 10 dimensions is $20^{10} \approx 10^{13}$ evaluations.  This exponential growth is called the **curse of dimensionality**, and it is why integrals in many dimensions are usually computed with Monte Carlo methods.

## Monte Carlo Integration

[Monte Carlo integration](https://en.wikipedia.org/wiki/Monte_Carlo_integration) is a method which computes integrals by taking a sum over random samples. 

\begin{equation}
\int_{a}^b f(x)\, dx = (b-a)\, \mathbb{E}_{U(a,b)} [f]
\end{equation}

Where $U(a,b)$ is the uniform distribution over the interval $[a,b]$. We can estimate this expected value by drawing samples from the distribution, and computing

\begin{equation}
\mathbb{E}_{U(a,b)} [f] \approx \frac{1}{N} \sum_{i=1}^N f(x_i), \quad x_i \sim U(a,b)
\end{equation}

Or, more generally, we might define an integral over a probability measure $\mu$

\begin{equation}
\int f(x)\, d\mu(x) = \mathbb{E}_{\mu} [f] \approx \frac{1}{N} \sum_{i=1}^N f(x_i), \quad x_i \sim \mu(x)
\end{equation}

The law of large numbers guarantees that the average converges to the integral as $N \to \infty$.  The size of the error comes from the variance: the average of $N$ independent samples $f(x_i)$ has variance $\sigma_f^2 / N$, where $\sigma_f^2$ is the variance of a single sample, so the typical error is $\sigma_f / \sqrt{N}$.  In high dimensions, this is typically the most effective way to integrate functions, as quadrature requires a number of samples that is exponential in terms of the dimension.

### Example

The classic example of Monte Carlo integration is computing $\pi$ using random samples.  Recall the area of the unit circle is $\pi$, and the area of the square $[-1,1]^2$ is $4$.  We sample points uniformly from this square, and compute how many fall in the unit circle - the expected ratio of points inside the circle to the total number of points is $\pi/4$.  [Monte Carlo Methods](../07_probability_statistics/monte_carlo.ipynb) has more on the error of Monte Carlo methods and on ways to reduce it.

In [ ]:
def compute_pi_mc(n=1000):
    x = np.random.rand(n,2)*2 - 1 # sampled in [-1,1] x [-1,1]
    r = np.linalg.norm(x, axis=1) # computes radius of each point
    return 4 * np.sum(r <= 1.) / n

In [ ]:
%time compute_pi_mc(10_000_000)

Ten million samples give only about three correct digits.  Each sample is 4 if the point lands in the circle, which happens with probability $p = \pi/4$, and 0 otherwise, so its standard deviation is $\sigma_f = 4\sqrt{p(1 - p)} \approx 1.64$, and the typical error with $N = 10^7$ samples is $\sigma_f/\sqrt{N} \approx 5 \times 10^{-4}$.  Each extra digit costs a hundred times more samples.  What makes Monte Carlo worthwhile is that the rate $N^{-1/2}$ is the same in any number of dimensions.

## Exercises

### Exercise 1

The trapezoidal rule is only second order in general, but periodic functions are an exception.  Compute $\int_0^{2\pi} e^{\cos x}\, dx = 2\pi I_0(1)$ (`scipy.special.i0` is $I_0$) with the trapezoidal rule on $n = 2, 4, 8, \dots, 64$ intervals, and plot the error.  How fast does it converge?  The Euler-Maclaurin formula for the error,
$$ T_h - \int_a^b f(x)\, dx = \frac{h^2}{12}\bigl(f'(b) - f'(a)\bigr) - \frac{h^4}{720}\bigl(f'''(b) - f'''(a)\bigr) + \cdots, $$
suggests why.

In [ ]:
# Your code here

### Exercise 2

The Laguerre polynomials are orthogonal for the weight $e^{-x}$ on $[0, \infty)$.  Their recurrence gives a symmetric tridiagonal matrix $J$ with diagonal entries $1, 3, 5, \dots, 2n - 1$ and off-diagonal entries $1, 2, \dots, n - 1$.

1. Modify `gauss_legendre` to compute the $n$-point Gauss-Laguerre rule.  The weight formula becomes $w_i = (\mathbf v_i)_1^2 \int_0^\infty e^{-x}\, dx = (\mathbf v_i)_1^2$.  Compare your nodes and weights with `scipy.special.roots_laguerre`.
2. Use your rule to compute $\int_0^\infty e^{-x} \sin x\, dx = 1/2$ and $\int_0^\infty e^{-x} \sqrt{x}\, dx = \sqrt{\pi}/2$ with $n = 2, 4, 8, \dots, 64$ points.  Why does the second one converge so much more slowly?

In [ ]:
# Your code here

### Exercise 3

Use `interpolatory_weights` to compute the weights of the Newton-Cotes rules with $n = 3, 5, 7, \dots, 25$ equally spaced nodes on $[-1, 1]$, including the end points.  Plot $\sum_i |w_i|$ against $n$, and compare with Gauss-Legendre, for which it is always 2.  Then integrate the Runge function $1/(1 + 25x^2)$ with both kinds of rule.  What goes wrong with Newton-Cotes, and how is it related to the Runge phenomenon in [Interpolation](interpolation.ipynb)?

In [ ]:
# Your code here

### Exercise 4

Write an adaptive Simpson's rule.  Given an interval $[a, b]$ and a tolerance $\varepsilon$, compare Simpson's rule $S(a, b)$ on the whole interval with $S(a, m) + S(m, b)$ on its two halves, where $m = (a + b)/2$.  If they differ by less than $15 \varepsilon$, accept $S(a, m) + S(m, b)$.  Otherwise, apply the method recursively to both halves, with tolerance $\varepsilon/2$ on each.

1. Where does the 15 come from?  (Hint: Simpson's rule is fourth order.)
2. Test your function on the peak $1/((x - 0.3)^2 + 10^{-4})$ from above, and plot the points where it evaluates $f$.  Make sure it never evaluates $f$ twice at the same point.

In [ ]:
# Your code here